# 04. State-of-the-Art Gradient Boosting Suite
Triển khai bộ ba thuật toán GBDT:
- `LightGBM Regressor` (Leaf-wise growth, early stopping trên Val 2024).
- `XGBoost Regressor` (Depth-wise growth, early stopping trên Val 2024).
- `CatBoost Regressor` (Oblivious trees, early stopping trên Val 2024).


In [1]:
import os
import time
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import lightgbm as lgb
import xgboost as xgb

try:
    import catboost as cb
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("CatBoost chưa cài đặt, tự động bỏ qua.")

BASE_DIR = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
if not os.path.exists(PROCESSED_DIR):
    PROCESSED_DIR = os.path.join(BASE_DIR, "data_processed")
ARTIFACTS_DIR = os.path.join(BASE_DIR, "artifacts")

def load_data(scenario):
    p_train = os.path.join(PROCESSED_DIR, f"dataset_{scenario}_train.parquet")
    p_val = os.path.join(PROCESSED_DIR, f"dataset_{scenario}_val.parquet")
    p_test = os.path.join(PROCESSED_DIR, f"dataset_{scenario}_test.parquet")
    if os.path.exists(p_train):
        return pd.read_parquet(p_train), pd.read_parquet(p_val), pd.read_parquet(p_test)
    return pd.read_csv(p_train.replace(".parquet", ".csv")), pd.read_csv(p_val.replace(".parquet", ".csv")), pd.read_csv(p_test.replace(".parquet", ".csv"))

def calculate_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    smape = np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100
    
    act_150 = (y_true > 150)
    pred_150 = (y_pred > 150)
    tp_150 = np.sum(act_150 & pred_150)
    recall_150 = tp_150 / (np.sum(act_150) + 1e-8)
    prec_150 = tp_150 / (np.sum(pred_150) + 1e-8)
    f1_150 = 2 * prec_150 * recall_150 / (prec_150 + recall_150 + 1e-8)
    return {
        "R2": float(r2), "MAE": float(mae), "RMSE": float(rmse),
        "sMAPE": float(smape), "Recall_150": float(recall_150),
        "Precision_150": float(prec_150), "F1_150": float(f1_150)
    }


### Huấn Luyện LightGBM, XGBoost, CatBoost


In [2]:
gbdt_results = []

for sc in ["1h", "24h"]:
    print(f"\n--- GBDT Suite: {sc.upper()} ---")
    train_df, val_df, test_df = load_data(sc)
    feat_cols = [c for c in train_df.columns if c not in ['time', 'year', 'target']]
    
    station_map = {sid: idx for idx, sid in enumerate(train_df['station_id'].unique())}
    X_train = train_df[feat_cols].copy()
    X_val = val_df[feat_cols].copy()
    X_test = test_df[feat_cols].copy()
    
    X_train['station_id'] = X_train['station_id'].map(station_map).astype(int)
    X_val['station_id'] = X_val['station_id'].map(station_map).fillna(0).astype(int)
    X_test['station_id'] = X_test['station_id'].map(station_map).fillna(0).astype(int)
    
    y_train = train_df['target'].values
    y_val = val_df['target'].values
    y_test = test_df['target'].values
    
    pred_df = test_df[['time', 'station_id', 'target']].copy().rename(columns={'target': 'actual'})
    
    # 1. LightGBM
    t0 = time.time()
    lgb_model = lgb.LGBMRegressor(
        n_estimators=500, learning_rate=0.05, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
    )
    lgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )
    time_lgb = time.time() - t0
    val_pred_lgb = lgb_model.predict(X_val)
    test_pred_lgb = lgb_model.predict(X_test)
    pred_df['pred_lightgbm'] = test_pred_lgb
    joblib.dump(lgb_model, f"{ARTIFACTS_DIR}/models/model_lightgbm_{sc}.joblib")
    
    m_val = calculate_metrics(y_val, val_pred_lgb)
    m_test = calculate_metrics(y_test, test_pred_lgb)
    gbdt_results.append({
        "scenario": sc, "family": "Gradient Boosting", "model": "LightGBM",
        "val_R2_2024": m_val["R2"], "val_MAE_2024": m_val["MAE"],
        "test_R2_2025": m_test["R2"], "test_MAE_2025": m_test["MAE"],
        "test_RMSE_2025": m_test["RMSE"], "test_sMAPE_2025": m_test["sMAPE"],
        "test_F1_150": m_test["F1_150"], "time_sec": time_lgb
    })
    print(f"LightGBM {sc} ({time_lgb:.1f}s): Test R2 = {m_test['R2']:.4f}, Test MAE = {m_test['MAE']:.2f}")
    
    # 2. XGBoost
    t0 = time.time()
    xgb_model = xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        n_jobs=-1, early_stopping_rounds=30
    )
    xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    time_xgb = time.time() - t0
    val_pred_xgb = xgb_model.predict(X_val)
    test_pred_xgb = xgb_model.predict(X_test)
    pred_df['pred_xgboost'] = test_pred_xgb
    joblib.dump(xgb_model, f"{ARTIFACTS_DIR}/models/model_xgboost_{sc}.joblib")
    
    m_val = calculate_metrics(y_val, val_pred_xgb)
    m_test = calculate_metrics(y_test, test_pred_xgb)
    gbdt_results.append({
        "scenario": sc, "family": "Gradient Boosting", "model": "XGBoost",
        "val_R2_2024": m_val["R2"], "val_MAE_2024": m_val["MAE"],
        "test_R2_2025": m_test["R2"], "test_MAE_2025": m_test["MAE"],
        "test_RMSE_2025": m_test["RMSE"], "test_sMAPE_2025": m_test["sMAPE"],
        "test_F1_150": m_test["F1_150"], "time_sec": time_xgb
    })
    print(f"XGBoost {sc} ({time_xgb:.1f}s): Test R2 = {m_test['R2']:.4f}, Test MAE = {m_test['MAE']:.2f}")
    
    # 3. CatBoost
    if HAS_CATBOOST:
        t0 = time.time()
        cb_model = cb.CatBoostRegressor(
            iterations=500, learning_rate=0.05, depth=6,
            random_seed=42, early_stopping_rounds=30, verbose=False, thread_count=-1
        )
        cb_model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)
        time_cb = time.time() - t0
        val_pred_cb = cb_model.predict(X_val)
        test_pred_cb = cb_model.predict(X_test)
        pred_df['pred_catboost'] = test_pred_cb
        joblib.dump(cb_model, f"{ARTIFACTS_DIR}/models/model_catboost_{sc}.joblib")
        
        m_val = calculate_metrics(y_val, val_pred_cb)
        m_test = calculate_metrics(y_test, test_pred_cb)
        gbdt_results.append({
            "scenario": sc, "family": "Gradient Boosting", "model": "CatBoost",
            "val_R2_2024": m_val["R2"], "val_MAE_2024": m_val["MAE"],
            "test_R2_2025": m_test["R2"], "test_MAE_2025": m_test["MAE"],
            "test_RMSE_2025": m_test["RMSE"], "test_sMAPE_2025": m_test["sMAPE"],
            "test_F1_150": m_test["F1_150"], "time_sec": time_cb
        })
        print(f"CatBoost {sc} ({time_cb:.1f}s): Test R2 = {m_test['R2']:.4f}, Test MAE = {m_test['MAE']:.2f}")
        
    pred_df.to_csv(f"{ARTIFACTS_DIR}/predictions/preds_{sc}_gbdt.csv", index=False)

df_gbdt_res = pd.DataFrame(gbdt_results)
df_gbdt_res.to_json(f"{ARTIFACTS_DIR}/metrics/metrics_gbdt.json", orient="records", indent=2)
df_gbdt_res[['scenario', 'model', 'val_R2_2024', 'test_R2_2025', 'test_MAE_2025', 'test_RMSE_2025', 'time_sec']]



--- GBDT Suite: 1H ---
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002723 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3812
[LightGBM] [Info] Number of data points in the train set: 111073, number of used features: 22
[LightGBM] [Info] Start training from score 54.880437


LightGBM 1h (1.0s): Test R2 = 0.9538, Test MAE = 3.13


XGBoost 1h (2.1s): Test R2 = 0.9538, Test MAE = 3.10


CatBoost 1h (3.2s): Test R2 = 0.9541, Test MAE = 3.13

--- GBDT Suite: 24H ---


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002214 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3047
[LightGBM] [Info] Number of data points in the train set: 110733, number of used features: 19
[LightGBM] [Info] Start training from score 54.801578


LightGBM 24h (0.4s): Test R2 = 0.6170, Test MAE = 11.33


XGBoost 24h (0.7s): Test R2 = 0.6159, Test MAE = 11.36


CatBoost 24h (1.4s): Test R2 = 0.6084, Test MAE = 11.40


,scenario,model,val_R2_2024,test_R2_2025,test_MAE_2025,test_RMSE_2025,time_sec
0,1h,LightGBM,0.909109,0.953796,3.130638,5.691877,1.043022
1,1h,XGBoost,0.909010,0.953831,3.100748,5.689672,2.128576
2,1h,CatBoost,0.907875,0.954057,3.131664,5.675783,3.167891
3,24h,LightGBM,0.549621,0.616972,11.325633,16.346828,0.410334
4,24h,XGBoost,0.556402,0.615870,11.361721,16.370327,0.671419
5,24h,CatBoost,0.553421,0.608415,11.400599,16.528408,1.447703


### Trích Xuất Feature Importance Của LightGBM


In [3]:
fi_records = []
for sc in ["1h", "24h"]:
    lgb_m = joblib.load(f"{ARTIFACTS_DIR}/models/model_lightgbm_{sc}.joblib")
    train_df, _, _ = load_data(sc)
    feat_cols = [c for c in train_df.columns if c not in ['time', 'year', 'target']]
    fi = pd.DataFrame({
        "feature": feat_cols,
        "importance": lgb_m.feature_importances_,
        "scenario": sc
    }).sort_values("importance", ascending=False)
    fi_records.append(fi)

df_fi = pd.concat(fi_records, ignore_index=True)
df_fi.to_csv(f"{ARTIFACTS_DIR}/metrics/feature_importance_lgb.csv", index=False)
print("Top 5 đặc trưng quan trọng nhất:")
print(df_fi.groupby('scenario').head(5))


Top 5 đặc trưng quan trọng nhất:


                    feature  importance scenario
0          target_aqi_lag_0        1594       1h
1                   diff_1h        1342       1h
2          target_aqi_lag_1         807       1h
3          target_aqi_lag_2         744       1h
4            temperature_2m         694       1h
22         target_aqi_lag_0         377      24h
23           future_temp_2m         302      24h
24  future_surface_pressure         271      24h
25               station_id         266      24h
26        future_wind_speed         247      24h
